# AI-Publisher GHCR Image Upload

Drive'daki Docker imaj arsivlerini (`.tar.gz`) GHCR'a (GitHub Container Registry) yukler.

### Akis:
1. Google Drive baglanir
2. `build_all_v2.sh` ile olusturulmus `.tar.gz` dosyalari taranir
3. Podman kurulur (daemonless, Colab'da calisir)
4. GHCR'a giris yapilir (GitHub PAT gerekli)
5. Her arsiv ayri ayri GHCR'a push edilir

**Gereken Colab Secret:** `GITHUB_PAT` (veya `GITHUB_TOKEN`, `GH_TOKEN`)

---
Her hucreyi sirayla calistirin.

## Hucre 1: Google Drive Baglantisi

In [ ]:
import os
import subprocess
import sys
import glob
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Konfigurasyon
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
GHCR_NAMESPACE = "ghcr.io/arda-avci"
IMAGE_PREFIX = "ai-publisher-"  # tar.gz icindeki image name prefix

print(f"Image dizini: {DRIVE_IMAGE_DIR}")
print(f"GHCR hedef: {GHCR_NAMESPACE}")

## Hucre 2: Mevcut Arsivleri Tara

In [ ]:
image_dir = Path(DRIVE_IMAGE_DIR)
if not image_dir.exists():
    print(f"HATA: {DRIVE_IMAGE_DIR} bulunamadi! Drive mount oldugunu dogrulayin.")
    sys.exit(1)

archives = sorted(image_dir.glob("*.tar.gz"))
print(f"\nBulunan arsiv sayisi: {len(archives)}\n")

for a in archives:
    size_mb = a.stat().st_size / (1024 * 1024)
    print(f"  {a.name}  ({size_mb:.0f} MB)")

if not archives:
    print("\nArsiv bulunamadi!")
    print(f"Beklenen dizin: {DRIVE_IMAGE_DIR}")
    print("\nOnce build_all_v2.sh calistirin.")

## Hucre 3: Podman Kurulumu

Podman daemonless container engine'dir. Docker imajlarini Docker daemon olmadan load/push edebilir.

In [ ]:
def run(cmd, check=True, capture=False):
    """Shell komutu calistir. Her zaman result objesi doner."""
    import subprocess
    result = subprocess.run(cmd, shell=True, text=True,
                           capture_output=capture)
    if capture:
        result._stdout = result.stdout.strip() if result.stdout else ""
    if check and result.returncode != 0:
        print(f"[HATA] Komut basarisiz: {cmd}")
        print(f"STDERR: {result.stderr}")
        sys.exit(1)
    return result

# Podman kurulu mu?
r = run("which podman", check=False, capture=True)
if r.returncode == 0:
    print(f"Podman mevcut: {r._stdout}")
    print(run("podman --version", capture=True)._stdout)
else:
    print("Podman kuruluyor...")
    run("apt-get update -qq")
    run("apt-get install -y -qq podman")
    print("Podman kuruldu.")
    print(run("podman --version", capture=True)._stdout)

## Hucre 4: GHCR Giris

Colab Secret'tan GitHub PAT okuyup GHCR'a giris yapar.
PAT'in `write:packages` izni olmali.

In [ ]:
from google.colab import userdata

gh_token = None
for key in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
    try:
        gh_token = userdata.get(key)
        if gh_token:
            print(f"Token bulundu: {key} (ilk 8: {gh_token[:8]}...)")
            break
    except Exception:
        pass

if not gh_token:
    print("HATA: GitHub PAT bulunamadi!")
    print("Colab Secret'a GITHUB_PAT, GITHUB_TOKEN veya GH_TOKEN ekleyin.")
    print("Islem: Colab UI'da Sag ust > Kilit ikonu > Secrets")
    sys.exit(1)

import base64, os, json

# Auth dosyasini podman config'ine yaz (Bearer token auth, --creds'den farkli)
_auth_b64 = base64.b64encode(f"Arda-Avci:{gh_token}".encode()).decode()
_auth_dir = os.path.expanduser("~/.config/containers")
os.makedirs(_auth_dir, exist_ok=True)
_auth_data = {"auths": {"ghcr.io": {"auth": _auth_b64}}}
with open(os.path.join(_auth_dir, "auth.json"), "w") as _f:
    json.dump(_auth_data, _f)
print(f"Auth dosyasi {_auth_dir}/auth.json yazildi.")

GHCR_AUTH = f"Arda-Avci:{gh_token}"  # skopeo --dest-creds icin yedek
print("Podman auth file ile push yapacak, --creds kullanilmayacak.")

## Hucre 5: Imajlari GHCR'a Push Et

Her `.tar.gz` dosyasini sirayla Podman'a yukler ve GHCR'a push eder.

Not: Buyuk imajlar (video modelleri 5-15 GB) push suresini uzatabilir.

In [ ]:
import time
import shutil
import shlex
import pathlib
import requests

TEMP_DIR = pathlib.Path("/content/tmp_images")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

def run_cap(cmd, timeout=None):
    import subprocess
    kwargs = dict(shell=True, text=True, capture_output=True)
    if timeout:
        kwargs["timeout"] = timeout
    try:
        r = subprocess.run(cmd, **kwargs)
    except subprocess.TimeoutExpired as e:
        class R:
            returncode = 124
            stdout = e.stdout or ""
            stderr = e.stderr or ""
            _stdout = stdout.strip()
        return R()
    r._stdout = r.stdout.strip() if r.stdout else ""
    return r

# Skopeo kurulu degilse kur
if not shutil.which("skopeo"):
    print("Skopeo kuruluyor...")
    run_cap("apt-get install -y -qq skopeo")
    if shutil.which("skopeo"):
        print("Skopeo kuruldu.")
    else:
        print("[WARN] Skopeo kurulamadi.")

# Crane kurulu degilse kur
crane_path = shutil.which("crane")
if not crane_path:
    print("Crane kuruluyor...")
    run_cap("wget -q https://github.com/google/go-containerregistry/releases/latest/download/go-containerregistry_Linux_x86_64.tar.gz -O /tmp/crane.tar.gz && tar -xzf /tmp/crane.tar.gz -C /usr/local/bin/ crane && chmod +x /usr/local/bin/crane")
    crane_path = shutil.which("crane")
    if crane_path:
        print("Crane kuruldu.")
    else:
        print("[WARN] Crane kurulamadi.")

# Crane auth: env var ile (crane push otomatik okur)
os.environ["CRANE_REGISTRY_TOKEN"] = gh_token

# --- DIAGNOSTIC: Token & GHCR kontrol ---
import requests as _req
_gh_headers = {"Authorization": f"Bearer {gh_token}", "Accept": "application/vnd.github.v3+json"}
# 1) Token gecerli mi?
_r = _req.get("https://api.github.com/user", headers=_gh_headers)
if _r.status_code == 200:
    print(f"[OK] Token gecerli. Kullanici: {_r.json()['login']}")
    _scopes = _r.headers.get('X-OAuth-Scopes', '(yok)').split(', ')
    print(f"[SCOPE] Token scope'lari: {_scopes}")
    print(f"[SCOPE] write:packages var mi: {'write:packages' in _scopes}")
else:
    print(f"[FAIL] Token gecersiz! HTTP {_r.status_code}: {_r.text[:100]}")
# 2) GHCR'da mevcut paketler neler?
_pkgs = []
_r = _req.get(f"https://api.github.com/user/packages?package_type=container", headers=_gh_headers)
if _r.status_code == 200:
    _pkgs = [p["name"] for p in _r.json()]
    print(f"[OK] GHCR mevcut paketler ({len(_pkgs)}): {_pkgs}")
elif _r.status_code == 403:
    print(f"[WARN] /user/packages API 403 (klasik PAT siniri, push'u etkilemez)")
    print(f"  Detay: {_r.text[:200]}")
else:
    print(f"[WARN] GHCR sorgu HTTP {_r.status_code}: {_r.text[:200]}")
# 3) Podman auth dogrulama
_r = _req.get("https://ghcr.io/v2/", headers=_gh_headers)
if _r.status_code == 200:
    print("[OK] GHCR API erisilebilir.")
else:
    print(f"[WARN] GHCR v2 API HTTP {_r.status_code}")
# 4) Mevcut bir paketi (varsa) gormeye calis
if _pkgs:
    _r = _req.get(f"https://ghcr.io/v2/arda-avci/{_pkgs[0]}/tags/list", headers=_gh_headers)
    print(f"  [{_pkgs[0]}] tags: HTTP {_r.status_code}" if _r.ok else f"  [{_pkgs[0]}] HTTP {_r.status_code}")
print("---")

# 5) Test push (podman auth file ile, --creds OLMADAN)
os.environ["CRANE_REGISTRY_TOKEN"] = gh_token
test_img = "ghcr.io/arda-avci/ai-publisher-test:test"
print("\nTest push (podman auth file ile)...")
run_cap("podman pull docker.io/library/alpine:latest", timeout=120)
r = run_cap(f"podman tag alpine:latest {test_img} && podman push {test_img}", timeout=180)
print(f"  Podman test push: RC={r.returncode}")
if r.returncode != 0:
    err_full = (r.stderr or r._stdout or '?')
    print(f"  HATA (ilk 500): {err_full[:500]}")
    if 'create_package' in err_full or 'permission_denied' in err_full:
        print("\n" + "=" * 50)
        print("SORUN: GHCR yeni paket olusturmaya izin vermiyor.")
        print("\nCOZUM 1: Token'da write:packages scope'u var mi?")
        print("  github.com/settings/tokens > token > Packages > write:packages")
        print("\nCOZUM 2: Yeni PAT olustur (klasik, tum izinler):")
        print("  github.com/settings/tokens > Generate new token > select all")
        print("  (write:packages ve repo dahil)")
        print("\nCOZUM 3: Fine-grained PAT:")
        print("  github.com/settings/tokens?type=beta")
        print("  -> Arda-Avci/AI-Publisher -> Packages: Write")
        print("=" * 50)
else:
    print(f"  [OK] Push calisiyor! Test imaji push edildi.")
    run_cap(f"podman rmi {test_img} 2>/dev/null")
run_cap("podman rmi alpine:latest 2>/dev/null")

total = len(archives)
success = 0
failed = []

for idx, archive_path in enumerate(archives, 1):
    model_name = archive_path.stem
    if model_name.endswith(".tar"):
        model_name = model_name[:-4]
    ghcr_image = f"{GHCR_NAMESPACE}/{IMAGE_PREFIX}{model_name}:latest"
    size_mb = archive_path.stat().st_size / (1024 * 1024)

    print("\n" + "=" * 60)
    print(f"[{idx}/{total}] {model_name}  ({size_mb:.0f} MB)")
    print(f"  -> {ghcr_image}")
    print("=" * 60)

    start = time.time()
    temp_tar = TEMP_DIR / f"{model_name}.tar"

    q_arc = shlex.quote(str(archive_path))
    q_tar = shlex.quote(str(temp_tar))

    # Decompress
    print(f"  [1/3] Decompress basladi...")
    r = run_cap(f"gunzip -c {q_arc} > {q_tar}")
    if r.returncode != 0:
        print(f"  [HATA] Decompress basarisiz! STDERR: {r.stderr}")
        failed.append(model_name)
        continue
    tar_size = temp_tar.stat().st_size / (1024 * 1024)
    print(f"  Decompress tamam. Temp: {tar_size:.0f} MB")

    pushed = False

    # Method 1: Podman load + push --creds
    print(f"  [2/3] Podman load basladi...")
    r = run_cap(f"podman load -i {q_tar}")
    if r.returncode == 0:
        loaded_name = None
        for line in r._stdout.split("\n"):
            if "Loaded image" in line:
                loaded_name = line.split(":", 1)[-1].strip()
                break
        if not loaded_name:
            loaded_name = f"localhost/{IMAGE_PREFIX}{model_name}:latest"
        print(f"  Load: {loaded_name}")

        for attempt in range(3):
            print(f"  [3/3] Podman push (deneme {attempt+1}/3)...")
            r = run_cap(f"podman tag {loaded_name} {ghcr_image} && podman push {ghcr_image}", timeout=1800)
            if r.returncode == 0:
                elapsed = time.time() - start
                print(f"  [OK] Podman push tamam. ({elapsed:.0f}s)")
                pushed = True
                break
            print(f"  [WARN] Podman push deneme {attempt+1} basarisiz. RC={r.returncode}")
            err = r.stderr or r._stdout or "(bos)"
            print(f"    Son 300: {err[-300:]}")
            if attempt < 2:
                time.sleep(10)
        run_cap(f"podman rmi {loaded_name} 2>/dev/null")
        run_cap(f"podman rmi {ghcr_image} 2>/dev/null")
    else:
        print(f"  [WARN] Podman load basarisiz. RC={r.returncode}")
        print(f"    STDERR: {(r.stderr or r._stdout or '(bos)')[:300]}")

    if pushed:
        temp_tar.unlink(missing_ok=True)
        success += 1
        continue

    # Method 2: Skopeo copy --dest-creds
    if shutil.which("skopeo"):
        print(f"  [2/3] Skopeo copy deneniyor...")
        r = run_cap(f"skopeo copy --dest-creds={GHCR_AUTH} docker-archive:{q_tar} docker://{ghcr_image}", timeout=1800)
        if r.returncode == 0:
            elapsed = time.time() - start
            print(f"  [OK] Skopeo push tamam. ({elapsed:.0f}s)")
            temp_tar.unlink(missing_ok=True)
            success += 1
            continue
        print(f"  [WARN] Skopeo basarisiz. RC={r.returncode}")
        print(f"    STDERR: {(r.stderr or '(bos)')[:300]}")
    else:
        print(f"  [SKIP] Skopeo mevcut degil.")

    # Method 3: crane push
    if crane_path:
        print(f"  [2/3] Crane push deneniyor...")
        r = run_cap(f"crane push {q_tar} {ghcr_image}", timeout=1800)
        if r.returncode == 0:
            elapsed = time.time() - start
            print(f"  [OK] Crane push tamam. ({elapsed:.0f}s)")
            temp_tar.unlink(missing_ok=True)
            success += 1
            continue
        print(f"  [WARN] Crane basarisiz. RC={r.returncode}")
        print(f"    STDERR: {(r.stderr or '(bos)')[:300]}")
    else:
        print(f"  [SKIP] Crane mevcut degil.")

    print(f"  [FAIL] {model_name} - tum yontemler basarisiz!")
    failed.append(model_name)
    temp_tar.unlink(missing_ok=True)

## Hucre 6: Ozet

In [ ]:
print("\n" + "=" * 60)
print("SONUC")
print("=" * 60)
print(f"  Basarili: {success}/{total}")

if failed:
    print(f"\n  Basarisiz ({len(failed)}):")
    for f in failed:
        print(f"    - {f}")
else:
    print("\n  Tum imajlar GHCR'a yuklendi.")

print("\nDogrulama:")
print(f"  https://github.com/Arda-Avci/AI-Publisher/pkgs/container/")
print(f"  veya: https://github.com/settings/packages")

## Hucre 7: (Opsiyonel) Dogrulama

GHCR'a push edilen imajlari Podman ile cekip listeler.

In [ ]:
import json

test_model = archives[0].stem if archives else "cogvideox"
if test_model.endswith(".tar"):
    test_model = test_model[:-4]
test_image = f"{GHCR_NAMESPACE}/{IMAGE_PREFIX}{test_model}:latest"

print(f"Test imaji cekiliyor: {test_image}")
result = run(f"podman pull {test_image}", check=False)
if result.returncode == 0:
    print(f"[OK] {test_image} basariyla cekildi.")
    run(f"podman rmi {test_image} 2>/dev/null", check=False)
else:
    print(f"[WARN] Pull basarisiz. Image GHCR'da goruntulenemiyor olabilir.")
    print("Gorunurluk: GitHub > Packages > package > Package settings > Change visibility > Public")

---

### Sorun Giderme

| Sorun | Cozum |
|-------|-------|
| `podman: command not found` | Hucre 3 podman kurulumu basarisiz. `apt-cache search podman` ile paket adini dogrulayin. |
| `denied: permission` | GitHub PAT'in `write:packages` izni yok. PAT'i guncelleyin. |
| `unauthorized` | Token gecersiz veya süresi dolmus. Yeni PAT olusturun. |
| Disk dolu | Colab Runtime > Disconnect and delete runtime > Yeniden baslatip sadece ghcr upload yapin. |
| `tar.gz` dosyasi bozuk | Dosyayi Drive'dan silip `build_all_v2.sh` ile yeniden olusturun. |
| Image cekilemiyor | GitHub > Packages > `Arda-Avci/ai-publisher-{model}` > **Make public** |